In [1]:
pip install transformers datasets scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [5]:
dataset = load_dataset('csv', data_files={'train': './Cleaned_TAMIL_DS.csv'}, split='train')
dataset = dataset.train_test_split(test_size=0.2)  

In [6]:
###Desktop/MULTI_LINGUAL_PROJECT/TAMIL/Cleaned_TAMIL_DS.csv

In [8]:
model_checkpoint = "xlm-roberta-base" 
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [9]:
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=256)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map: 100%|████████████████████████| 6830/6830 [00:00<00:00, 24450.91 examples/s]


In [10]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [11]:
num_labels = len(set(dataset['train']['label'])) 
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

In [14]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",        
    save_strategy="epoch",              
    learning_rate=2e-5,
    per_device_train_batch_size=4,     
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,      
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,        
)

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

/var/folders/hh/c39hlsnx5wb0yx5f42tpn3mm0000gn/T/ipykernel_12063/941576755.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [16]:
trainer.train()

eval_results = trainer.evaluate()
print("Evaluation Results:", eval_results)

/opt/anaconda3/envs/tamil_nlp/lib/python3.9/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.722300,0.654265,0.739385,0.688864,0.692972,0.739385
2,0.763000,0.649004,0.738799,0.715972,0.707013,0.738799
3,0.576600,0.658038,0.739385,0.710710,0.715537,0.739385


/opt/anaconda3/envs/tamil_nlp/lib/python3.9/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/envs/tamil_nlp/lib/python3.9/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/envs/tamil_nlp/lib/python3.9/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Evaluation Results: {'eval_loss': 0.6542651057243347, 'eval_accuracy': 0.739385065885798, 'eval_f1': 0.6888641909549067, 'eval_precision': 0.692971639819812, 'eval_recall': 0.739385065885798, 'eval_runtime': 141.2005, 'eval_samples_per_second': 48.371, 'eval_steps_per_second': 12.096, 'epoch': 3.0}
